# Sesión 5: Consultas Agrupadas (Parte II)
## HAVING: Filtrado de Resultados Agrupados

**Módulo:** Fundamentos de Programación Python para el Análisis de Datos

**Contenido:**
- Cláusula HAVING: Aplicar condiciones sobre datos agrupados
- Diferencias entre WHERE y HAVING
- Combinación de WHERE + HAVING para optimización
- Errores comunes y buenas prácticas

## Configuración Inicial

In [ ]:
import sqlite3
import pandas as pd
import numpy as np

# Crear conexión
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

print("✓ Conexión SQLite establecida")

## SLIDE 3: Desafío Inicial

**Contexto:** Institución de formación técnica con programas nacionales.

**Solicitud:**
1. Promedios de calificación por programa
2. Modalidades con mejor rendimiento
3. Programas con al menos X estudiantes Y promedio superior a Y

**Pregunta:** ¿Cómo filtrar grupos basado en agregaciones?

In [ ]:
# Crear tabla con más datos que Sesión 4
cursor.execute('''
CREATE TABLE programas_evaluaciones (
    estudiante VARCHAR(50),
    programa VARCHAR(50),
    modalidad VARCHAR(20),
    calificacion DECIMAL(3, 1)
)
''')

# Insertar datos más completos
datos = [
    # Python Avanzado
    ('Ana García', 'Python Avanzado', 'Presencial', 8.5),
    ('Bruno López', 'Python Avanzado', 'Presencial', 9.0),
    ('Carlos Martín', 'Python Avanzado', 'Virtual', 7.5),
    ('Gabriel López', 'Python Avanzado', 'Virtual', 8.0),
    ('María Pérez', 'Python Avanzado', 'Presencial', 8.5),
    
    # SQL Empresarial
    ('Diana Ruiz', 'SQL Empresarial', 'Presencial', 8.0),
    ('Elena Sánchez', 'SQL Empresarial', 'Virtual', 8.5),
    ('Fiona Chen', 'SQL Empresarial', 'Virtual', 7.0),
    
    # Cloud Computing
    ('Héctor Ruiz', 'Cloud Computing', 'Presencial', 9.5),
    ('Isabel Martín', 'Cloud Computing', 'Presencial', 8.5),
    ('Javier García', 'Cloud Computing', 'Virtual', 7.5),
    ('Kevin López', 'Cloud Computing', 'Presencial', 9.0),
    ('Laura Martín', 'Cloud Computing', 'Virtual', 8.0),
    
    # Big Data
    ('Karen López', 'Big Data', 'Virtual', 9.0),
    ('Luis Pérez', 'Big Data', 'Virtual', 8.5),
    
    # DevOps (nuevo programa)
    ('Marcos García', 'DevOps', 'Presencial', 7.0),
    ('Natalia López', 'DevOps', 'Presencial', 6.5),
]

cursor.executemany(
    'INSERT INTO programas_evaluaciones VALUES (?, ?, ?, ?)',
    datos
)

conn.commit()
print(f"✓ Tabla 'programas_evaluaciones' creada con {len(datos)} registros\n")

df = pd.read_sql('SELECT * FROM programas_evaluaciones', conn)
print("Primeros 10 registros:")
print(df.head(10))

## SLIDE 4-5: HAVING - Concepto y Sintaxis

**¿Qué es HAVING?** Cláusula que aplica condiciones sobre datos YA AGRUPADOS.

**Sintaxis:**
```sql
SELECT columna, COUNT(*), AVG(calificacion)
FROM tabla
GROUP BY columna
HAVING AVG(calificacion) > 6.5
```

**Diferencia clave:** WHERE filtra ANTES de agrupar. HAVING filtra DESPUÉS.

In [ ]:
print("="*70)
print("SLIDE 5: Ejemplo Básico de HAVING")
print("="*70 + "\n")

# Ejemplo: Programas con promedio > 8.0
query = '''
SELECT 
    programa,
    COUNT(*) as cantidad,
    ROUND(AVG(calificacion), 2) as promedio
FROM programas_evaluaciones
GROUP BY programa
HAVING AVG(calificacion) > 8.0
ORDER BY promedio DESC
'''

print("Consulta: Programas con promedio > 8.0")
print(query)
print("\nResultado:")
df_having = pd.read_sql(query, conn)
print(df_having.to_string())
print(f"\n✓ Solo {len(df_having)} programas tienen promedio > 8.0")

## SLIDE 6: Importancia de HAVING en Contextos Reales

In [ ]:
print("\n" + "="*70)
print("SLIDE 6: Casos de Uso Reales de HAVING")
print("="*70 + "\n")

print("""Contextos donde HAVING es esencial:
1. Educación: Programas con promedio >= 6.0 para acreditación
2. Comercio: Productos vendidos >= 100 unidades
3. Salud: Pacientes con edad promedio > 65 años
4. Recursos Humanos: Departamentos con más de 10 empleados
5. Análisis de datos: Grupos con desviación > umbral\n""")

# Ejemplo comercial
print("-"*70)
print("Ejemplo: Programas con al menos 4 estudiantes")
print("-"*70 + "\n")

query_real = '''
SELECT 
    programa,
    COUNT(*) as estudiantes,
    ROUND(AVG(calificacion), 2) as promedio
FROM programas_evaluaciones
GROUP BY programa
HAVING COUNT(*) >= 4
ORDER BY estudiantes DESC
'''

df_real = pd.read_sql(query_real, conn)
print(df_real.to_string())
print(f"\nInterpretación: Solo programas con 4+ estudiantes están en reporte")

## SLIDE 7: Diferencias WHERE vs HAVING

In [ ]:
print("\n" + "="*70)
print("SLIDE 7: WHERE vs HAVING - Comparación")
print("="*70 + "\n")

comparacion = {
    'Característica': [
        'Se ejecuta',
        'Filtra',
        'Usa agregación',
        'Tipo de condición',
        'Eficiencia',
        'Ejemplo'
    ],
    'WHERE': [
        'Antes de GROUP BY',
        'Filas individuales',
        'No',
        'calificacion > 7.0',
        'Filtra primero (mejor)',
        'WHERE modalidad = "Presencial"'
    ],
    'HAVING': [
        'Después de GROUP BY',
        'Grupos de resultados',
        'Sí',
        'AVG(calificacion) > 8.0',
        'Filtra después (procesa más datos)',
        'HAVING COUNT(*) > 5'
    ]
}

df_comp = pd.DataFrame(comparacion)
print(df_comp.to_string(index=False))

print("\n" + "="*70)

In [ ]:
# Demostración práctica: WHERE vs HAVING
print("\n" + "-"*70)
print("Demostración Práctica")
print("-"*70 + "\n")

# Query 1: WHERE (filtra ANTES)
print("✓ QUERY 1: WHERE - Filtra antes de agrupar")
query_where = '''
SELECT 
    programa,
    COUNT(*) as total
FROM programas_evaluaciones
WHERE calificacion >= 8.0
GROUP BY programa
'''
print(query_where)
df_where = pd.read_sql(query_where, conn)
print("\nResultado (solo estudiantes con cal >= 8.0):")
print(df_where.to_string())

# Query 2: HAVING (filtra DESPUÉS)
print("\n✓ QUERY 2: HAVING - Filtra después de agrupar")
query_having = '''
SELECT 
    programa,
    COUNT(*) as total,
    ROUND(AVG(calificacion), 2) as promedio
FROM programas_evaluaciones
GROUP BY programa
HAVING AVG(calificacion) >= 8.0
'''
print(query_having)
df_having_demo = pd.read_sql(query_having, conn)
print("\nResultado (grupos con promedio >= 8.0):")
print(df_having_demo.to_string())

print("\n💡 Diferencia: WHERE excluye individuos. HAVING excluye grupos.")

## SLIDE 8: Buenas Prácticas

In [ ]:
print("\n" + "="*70)
print("SLIDE 8: Buenas Prácticas con HAVING")
print("="*70 + "\n")

print("✓ BUENA PRÁCTICA 1: Usar alias (AS) para claridad")
query_alias = '''
SELECT 
    programa,
    COUNT(*) AS cantidad_estudiantes,
    ROUND(AVG(calificacion), 2) AS promedio_calificaciones
FROM programas_evaluaciones
GROUP BY programa
HAVING AVG(calificacion) > 7.5
'''
print(query_alias)
print("\nResultado:")
df_alias = pd.read_sql(query_alias, conn)
print(df_alias.to_string())
print("\n✓ Los alias hacen el resultado más legible")

In [ ]:
print("\n✓ BUENA PRÁCTICA 2: Combinar WHERE + HAVING para optimización")
query_combo = '''
SELECT 
    programa,
    modalidad,
    COUNT(*) AS cantidad,
    ROUND(AVG(calificacion), 2) AS promedio
FROM programas_evaluaciones
WHERE calificacion >= 7.0  -- Filtra datos antes de agrupar
GROUP BY programa, modalidad
HAVING COUNT(*) >= 2  -- Filtra grupos después
ORDER BY programa, promedio DESC
'''
print(query_combo)
print("\nResultado:")
df_combo = pd.read_sql(query_combo, conn)
print(df_combo.to_string())
print("\n✓ WHERE reduce volumen primero, HAVING filtra resultado final")

## SLIDE 9: Errores Comunes

In [ ]:
print("\n" + "="*70)
print("SLIDE 9: Errores Comunes con HAVING")
print("="*70 + "\n")

print("✗ ERROR 1: Usar HAVING sin GROUP BY")
print("-"*70)
print("""Código incorrecto:
SELECT programa, AVG(calificacion)
HAVING AVG(calificacion) > 8.0
-- Error: HAVING sin GROUP BY

Solución:
SELECT programa, AVG(calificacion)
FROM tabla
GROUP BY programa
HAVING AVG(calificacion) > 8.0
""")

print("\n✗ ERROR 2: Omitir agregación en HAVING")
print("-"*70)
print("""Código incorrecto:
SELECT programa, COUNT(*)
FROM tabla
GROUP BY programa
HAVING programa = 'Python'  -- Esto es WHERE, no HAVING

Solución:
SELECT programa, COUNT(*)
FROM tabla
GROUP BY programa
HAVING COUNT(*) > 5  -- Uso de agregación
""")

print("\n✗ ERROR 3: Usar alias no definido")
print("-"*70)
print("""Código incorrecto:
SELECT programa, COUNT(*) AS qty
FROM tabla
GROUP BY programa
HAVING qty > 5  -- qty no se conoce en HAVING (en algunos motores)

Solución:
SELECT programa, COUNT(*) AS qty
FROM tabla
GROUP BY programa
HAVING COUNT(*) > 5  -- Usar función completa
""")

## SLIDE 12-15: Actividad Guiada - Condiciones en Datos Agrupados

In [ ]:
print("\n" + "="*70)
print("ACTIVIDAD GUIADA: Aplicando HAVING")
print("="*70 + "\n")

print("📋 Contexto: Analista de datos en institución técnica.")
print("Necesitas reportes segmentados con condiciones en agregaciones.\n")

# Consulta base
print("✓ CONSULTA BASE: Todos los programas con estadísticas")
print("-"*70)

query_base = '''
SELECT 
    programa,
    COUNT(*) as cantidad_estudiantes,
    ROUND(AVG(calificacion), 2) as promedio_calificaciones,
    MIN(calificacion) as minima,
    MAX(calificacion) as maxima
FROM programas_evaluaciones
GROUP BY programa
ORDER BY promedio_calificaciones DESC
'''

print("SQL:")
print(query_base)
print("\nResultado:")
df_base = pd.read_sql(query_base, conn)
print(df_base.to_string())

In [ ]:
# Filtro 1: Programas con promedio > 6.5
print("\n✓ FILTRO 1: Mostrar programas con promedio > 6.5")
print("-"*70)

q1 = '''
SELECT 
    programa,
    COUNT(*) as cantidad,
    ROUND(AVG(calificacion), 2) as promedio
FROM programas_evaluaciones
GROUP BY programa
HAVING AVG(calificacion) > 6.5
ORDER BY promedio DESC
'''

print("SQL:")
print(q1)
print("\nResultado:")
df_f1 = pd.read_sql(q1, conn)
print(df_f1.to_string())
print(f"\n✓ {len(df_f1)} programas cumplen con promedio > 6.5")

In [ ]:
# Filtro 2: Programas con al menos 4 estudiantes
print("\n✓ FILTRO 2: Mostrar programas con al menos 4 estudiantes")
print("-"*70)

q2 = '''
SELECT 
    programa,
    COUNT(*) as cantidad,
    ROUND(AVG(calificacion), 2) as promedio
FROM programas_evaluaciones
GROUP BY programa
HAVING COUNT(*) >= 4
ORDER BY cantidad DESC
'''

print("SQL:")
print(q2)
print("\nResultado:")
df_f2 = pd.read_sql(q2, conn)
print(df_f2.to_string())
print(f"\n✓ {len(df_f2)} programas tienen 4+ estudiantes")

In [ ]:
# Filtro 3: Combinar ambas condiciones
print("\n✓ FILTRO 3: Combinar ambas condiciones (4+ estudiantes Y promedio > 8.0)")
print("-"*70)

q3 = '''
SELECT 
    programa,
    COUNT(*) as cantidad,
    ROUND(AVG(calificacion), 2) as promedio
FROM programas_evaluaciones
GROUP BY programa
HAVING COUNT(*) >= 4 AND AVG(calificacion) > 8.0
ORDER BY promedio DESC
'''

print("SQL:")
print(q3)
print("\nResultado:")
df_f3 = pd.read_sql(q3, conn)
print(df_f3.to_string())
print(f"\n✓ {len(df_f3)} programas cumplen AMBAS condiciones")
print("   (4+ estudiantes Y promedio > 8.0)")

## SLIDE 16-19: Actividad Práctica Autónoma

In [ ]:
print("\n" + "="*70)
print("ACTIVIDAD PRÁCTICA AUTÓNOMA: Filtrado Avanzado")
print("="*70 + "\n")

print("📋 Contexto: Base de datos académica de institución técnica.")
print("Aplicar HAVING para evaluación de programas.\n")

# Ejercicio 1
print("Ejercicio 1: Programas con más de 4 estudiantes")
print("-"*70)

ej1 = '''
SELECT 
    programa,
    COUNT(*) as cantidad_estudiantes,
    ROUND(AVG(calificacion), 2) as promedio_calificaciones
FROM programas_evaluaciones
GROUP BY programa
HAVING COUNT(*) > 4
ORDER BY cantidad_estudiantes DESC
'''

df_ej1 = pd.read_sql(ej1, conn)
print("\nResultado:")
print(df_ej1.to_string())

In [ ]:
# Ejercicio 2
print("\nEjercicio 2: Modalidades con promedio >= 8.0")
print("-"*70)

ej2 = '''
SELECT 
    modalidad,
    COUNT(*) as cantidad,
    ROUND(AVG(calificacion), 2) as promedio
FROM programas_evaluaciones
GROUP BY modalidad
HAVING AVG(calificacion) >= 8.0
'''

df_ej2 = pd.read_sql(ej2, conn)
print("\nResultado:")
print(df_ej2.to_string())

In [ ]:
# Ejercicio 3
print("\nEjercicio 3: Programa x Modalidad (3+ estudiantes Y promedio >= 8.0)")
print("-"*70)

ej3 = '''
SELECT 
    programa,
    modalidad,
    COUNT(*) as cantidad,
    ROUND(AVG(calificacion), 2) as promedio,
    ROUND(MAX(calificacion) - MIN(calificacion), 2) as rango
FROM programas_evaluaciones
GROUP BY programa, modalidad
HAVING COUNT(*) >= 3 AND AVG(calificacion) >= 8.0
ORDER BY programa, promedio DESC
'''

df_ej3 = pd.read_sql(ej3, conn)
print("\nResultado:")
print(df_ej3.to_string() if len(df_ej3) > 0 else "No hay combinaciones que cumplan")

## SLIDE 20-22: Resumen y Preguntas de Cierre

In [ ]:
print("\n" + "="*70)
print("RESUMEN - SESIÓN 5")
print("="*70 + "\n")

resumen = """
✓ PROPÓSITO DE HAVING:
  - Filtrar DESPUÉS de agrupar (GROUP BY)
  - Aplicar condiciones sobre agregaciones
  - Seleccionar solo grupos que cumplen criterios

✓ DIFERENCIA CLAVE WHERE vs HAVING:
  - WHERE: antes de agrupar (filtra filas)
  - HAVING: después de agrupar (filtra grupos)

✓ SINTAXIS:
  SELECT columna, agregacion
  FROM tabla
  GROUP BY columna
  HAVING agregacion > condición

✓ COMBINACIÓN WHERE + HAVING:
  - WHERE reduce volumen de datos
  - HAVING filtra resultado final
  - Mejora rendimiento

✓ ERRORES A EVITAR:
  - HAVING sin GROUP BY
  - Confundir WHERE y HAVING
  - Omitir agregaciones en HAVING
"""

print(resumen)
print("="*70)

In [ ]:
print("\n❓ PREGUNTAS DE CIERRE:\n")

preguntas = [
    "1. ¿Cuál es la diferencia fundamental entre WHERE y HAVING?\n   WHERE filtra filas ANTES de agrupar. HAVING filtra DESPUÉS del GROUP BY.",
    
    "2. ¿Qué ocurre si se intenta usar HAVING sin GROUP BY?\n   Error o resultado inconsistente (depende del motor SQL).",
    
    "3. ¿Cómo combinar múltiples condiciones en HAVING?\n   Usar AND/OR: HAVING COUNT(*) > 5 AND AVG(nota) > 7",
    
    "4. ¿Por qué es importante combinar WHERE + HAVING?\n   WHERE reduce datos primero, mejora rendimiento significativamente.",
    
    "5. ¿Cuándo preferir HAVING sobre WHERE?\n   Cuando necesitas filtrar sobre agregaciones (COUNT, SUM, AVG, etc.)"
]

for p in preguntas:
    print(p)
    print()

## Cierre

In [ ]:
conn.close()
print("\n✓ Conexión cerrada")
print("\n¡Fin de la Sesión 5!")
print("\n📚 Próxima sesión: Subconsultas (Subqueries)")